<a href="https://colab.research.google.com/github/paumorales480/Cosas-que-hago-jiji/blob/main/Actualizaci%C3%B3n_asistentes_DAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:

#Limpiamos el entorno de trabajo
rm(list = ls())

In [54]:
#Cargamos las librerías necesarias! (Otro día tendré que hacer un archivo para explicar las librerías que más uso)
if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, srvyr, openxlsx, readr, janitor, dplyr, stringr, googlesheets4, writexl,stringi, tidyr)

Loading required package: pacman



In [55]:
# Extraer el ID del documento y construir la URL de exportación a CSV
sheet_id <- "1crPn_A5hIYGZeQgLyrASUtjBE2NxHUcBjy8_wK1wt74"
url_csv <- paste0("https://docs.google.com/spreadsheets/d/", sheet_id, "/export?format=csv")

concentrado <- read.csv(url_csv) %>%
  clean_names()


In [57]:
asistentes <- concentrado %>%
  select(nombre_de_la_institucion, tipo_de_asistente, cargo, nombre_s, apellido_paterno, apellido_materno,
         cargo_1, nombre_s_2, apellido_paterno_1, apellido_materno_1) %>%
  mutate(across(c(nombre_s, apellido_paterno, apellido_materno,
                   nombre_s_2, apellido_paterno_1, apellido_materno_1),
                ~ str_squish(toupper(.x)))) %>%
  mutate(nombre_1 = paste(apellido_paterno, apellido_materno, nombre_s),
         nombre_2 = paste(apellido_paterno_1, apellido_materno_1, nombre_s_2)) %>%
  rename(cargo_2 = cargo_1,
         cargo_1 = cargo) %>%
  select(nombre_de_la_institucion, tipo_de_asistente, cargo_1, nombre_1, cargo_2, nombre_2) %>%
  pivot_longer(
    cols = c(cargo_1, nombre_1, cargo_2, nombre_2),
    names_to = c(".value", "persona"),
    names_sep = "_"
  ) %>%
  filter(!is.na(nombre) & trimws(nombre) != "" & nombre != "NA NA NA") %>%
  select(-persona) %>%
  mutate(nombre_normalizado = stri_trans_general(nombre, "Latin-ASCII")) %>%
  distinct(nombre_normalizado, .keep_all = TRUE) %>%
  select(-nombre_normalizado)

In [58]:
nrow(asistentes)
write_xlsx(asistentes, "asistentes.xlsx")

[1] 268

In [52]:
#OJITO


# Se abre el doc
gs4_auth()

sheet_url <- "https://docs.google.com/spreadsheets/d/1crPn_A5hIYGZeQgLyrASUtjBE2NxHUcBjy8_wK1wt74/edit"

# Se crea una nueva ventana con los asistentes actualizados! jiji
sheet_write(asistentes, ss = sheet_url, sheet = "asistentes_limpio_n")

The googlesheets4 package is requesting access to your Google account.
Enter '1' to start a new auth process or select a pre-authorized account.
1: Send me to the browser for a new auth process.
2: informes.institucionales256@gmail.com


Selection: 2


✔ Writing to "Formulario de registro_DIAUI2026 (Respuestas)".

✔ Writing to sheet asistentes_limpio.

